```
========================================================================
                      [ NOTICIA / TEXTO COMPLETO ]
========================================================================
                                  |
                                  v
                            [ MEC GEN ]
                                  |
      +---------------------------+---------------------------+
      | (Extrae párrafos individuales: Párrafo 1, Párrafo 2, Párrafo 3...)
      v
========================================================================
                   [ MODELO CLASIFICADOR PRINCIPAL ] 
                      (Detección de Falsedad)
========================================================================
                                  |
      +---------------------------+---------------------------+
      |                                                       |
      v                                                       v
[ ANÁLISIS GENERAL ]                                 [ ANÁLISIS POR PÁRRAFO ]
- Noticia en General: FALSA                          - P1: Falso (Score: 0.8)
                                                     - P2: Falso (Score: 0.7)
                                                     - P3: Verdadero (Score: 0.65)
                                                              |
                                                              v
                                             (Filtro: Pasan los párrafos 
                                              con el SCORE MÁS ALTO)
                                                              |
                                  +---------------------------+
                                  |
                                  v
                [ PÁRRAFOS FILTRADOS + NOTICIA COMPLETA ]
                                  |
      +---------------------------+---------------------------+
      |                                                       |
      v                                                       v
=============================                   =============================
[   MODELO CLASIFICADOR     ]                   [   MODELO CLASIFICADOR     ]
[     SENSACIONALISMO       ]                   [       REDUNDANCIA         ]
=============================                   =============================
      |                                                       |
      +-> Texto General:                                      +-> Texto General:
      |   [ Sensacionalista ]                                 |   [ Redundante ]
      |                                                       |
      +-> Párrafo 1 (P1):                                     +-> Párrafo 1 (P1):
      |   [ NO Sensacional ] (Score: 0.13)                    |   [ NO Redundante ] (Score: 0.3)
      |                                                       |
      +-> Párrafo 3 (P3):                                     +-> Párrafo 3 (P3):
          [ Sensacionalista ] (Score: 0.9)                        [ Redundante ] (Score: 0.8)
```


# 📦 1. Instalación de Dependencias y Descarga de Recursos NLP
En esta celda se instalan y descargan las librerías necesarias para ejecutar el pipeline de modelos en **Google Colab** o en un entorno local.


In [ ]:
# Instalación de librerías para PyTorch, Transformers, SentenceTransformers, SpaCy y NLTK
!pip install -q torch transformers sentence-transformers spacy nltk pandas numpy scikit-learn polars scipy

import nltk
import spacy

print("📥 Descargando recursos de NLTK y modelo de spaCy en español...")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

try:
    nlp = spacy.load("es_core_news_sm")
except Exception:
    import os
    os.system("python -m spacy download es_core_news_sm")
    nlp = spacy.load("es_core_news_sm")

print("✅ Entorno preparado correctamente con todas las librerías necesarias.")


# ⚙️ 2. Importación de Módulos e Inicialización de Dispositivo (GPU/CPU)
Importación de módulos requeridos y verificación del dispositivo de cómputo (CUDA GPU o CPU).


In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
from sentence_transformers import SentenceTransformer, util

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"💻 Dispositivo seleccionado para el pipeline: {device}")
if device.type == 'cuda':
    print(f"🚀 GPU Activa: {torch.cuda.get_device_name(0)}")


# 🤖 3. Carga e Inicialización de los Modelos del Pipeline
En esta sección cargamos los 3 modelos principales que integran el pipeline de investigación:
1. **SaBERT**: `VerificadoProfesional/SaBERT-Spanish-Fake-News` (Modelo clasificador principal de falsedad/veracidad).
2. **Bert Spanish Sensationalism**: `JJNeila/bert-spanish-sensationalism-oss` (Modelo detector de sensacionalismo).
3. **SBERT**: `paraphrase-multilingual-mpnet-base-v2` (Modelo de embeddings para análisis de redundancia).


In [ ]:
print("⏳ Cargando Modelo 1: SaBERT (Detección de Falsedad)...")
sabert_tokenizer = BertTokenizer.from_pretrained("VerificadoProfesional/SaBERT-Spanish-Fake-News")
sabert_model = BertForSequenceClassification.from_pretrained("VerificadoProfesional/SaBERT-Spanish-Fake-News").to(device)
sabert_model.eval()
print("✅ SaBERT cargado exitosamente.")

print("\n⏳ Cargando Modelo 2: Clasificador de Sensacionalismo...")
sensacionalismo_classifier = pipeline(
    "text-classification",
    model="JJNeila/bert-spanish-sensationalism-oss",
    tokenizer="JJNeila/bert-spanish-sensationalism-oss",
    device=0 if torch.cuda.is_available() else -1,
    top_k=None
)
print("✅ Clasificador de Sensacionalismo cargado exitosamente.")

print("\n⏳ Cargando Modelo 3: SentenceTransformers (SBERT para Redundancia)...")
sbert_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2', device=str(device))
print("✅ SBERT cargado exitosamente.")


# 🛠️ 4. Definición de Funciones de Procesamiento (MEC GEN, SaBERT, Sensacionalismo y Redundancia)
Definición de submódulos:
- `mec_gen_extraer_parrafos`: Descompone la noticia en párrafos u oraciones individuales.
- `evaluar_sabert_falsedad`: Evalúa falsedad/veracidad y probabilidades con SaBERT.
- `evaluar_sensacionalismo`: Mide el grado de sensacionalismo del texto.
- `evaluar_redundancia`: Determina similitud semántica y redundancia informativa.


In [ ]:
def mec_gen_extraer_parrafos(texto):
    """
    Extracción e identificación de párrafos u oraciones individuales (MEC GEN).
    """
    if not isinstance(texto, str) or not texto.strip():
        return []
    
    lineas = [p.strip() for p in texto.split("\n") if len(p.strip()) > 10]
    if len(lineas) >= 2:
        return lineas
    
    doc = nlp(texto)
    oraciones = [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 15]
    if len(oraciones) >= 2:
        return oraciones
    
    fragmentos = [frag.strip() for frag in texto.replace('\n', ' ').split('.') if len(frag.strip()) > 15]
    return fragmentos if fragmentos else [texto.strip()]

def evaluar_sabert_falsedad(texto):
    """
    Clasifica si un texto o párrafo es FALSO o VERDADERO usando SaBERT.
    """
    inputs = sabert_tokenizer(str(texto), return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = sabert_model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).squeeze().tolist()
    if isinstance(probs, float):
        probs = [probs, 1.0 - probs]
        
    prob_fake = float(probs[0])
    prob_true = float(probs[1])
    etiqueta = "FALSO" if prob_fake >= prob_true else "VERDADERO"
    return etiqueta, prob_fake, prob_true

def evaluar_sensacionalismo(texto):
    """
    Clasifica si un texto o párrafo es Sensacionalista usando Bert Spanish Sensationalism.
    """
    res = sensacionalismo_classifier(str(texto))[0]
    score_sens = next(item["score"] for item in res if item["label"] in ["LABEL_1", "Sensacionalista"])
    etiqueta = "Sensacionalista" if score_sens >= 0.50 else "NO Sensacional"
    return etiqueta, float(score_sens)

def evaluar_redundancia(texto_general, lista_parrafos, umbral=0.70):
    """
    Calcula la redundancia global de la noticia y la redundancia relativa de cada párrafo con SBERT.
    """
    if len(lista_parrafos) < 2:
        return {
            "global_label": "NO Redundante",
            "global_score": 0.0,
            "parrafos_scores": {i: (0.0, "NO Redundante") for i in range(len(lista_parrafos))}
        }
    
    embeddings = sbert_model.encode(lista_parrafos, convert_to_tensor=True)
    matriz_sim = util.cos_sim(embeddings, embeddings).cpu().numpy()
    
    n = len(lista_parrafos)
    similitudes_maximas_parrafo = []
    detalles_parrafos = {}
    
    for i in range(n):
        sims_otros = [matriz_sim[i][j] for j in range(n) if i != j]
        max_sim = float(np.max(sims_otros)) if sims_otros else 0.0
        lbl_p = "Redundante" if max_sim >= umbral else "NO Redundante"
        detalles_parrafos[i] = (max_sim, lbl_p)
        similitudes_maximas_parrafo.append(max_sim)
        
    score_global = float(np.mean(similitudes_maximas_parrafo)) if similitudes_maximas_parrafo else 0.0
    label_global = "Redundante" if score_global >= umbral else "NO Redundante"
    
    return {
        "global_label": label_global,
        "global_score": score_global,
        "parrafos_scores": detalles_parrafos
    }


# 🚀 5. Función Principal del Pipeline (`ejecutar_pipeline_tesis`)
Esta función integra todos los módulos y genera los registros (logs con emojis) para seguir el proceso completo desde la noticia cruda hasta la clasificación final.


In [ ]:
def ejecutar_pipeline_tesis(noticia_texto, top_k_filtro=2, umbral_score_falsedad=0.50):
    print("=" * 80)
    print(" 📰 [ NOTICIA / TEXTO COMPLETO RECIBIDO ]")
    print("=" * 80)
    print(f"{noticia_texto.strip()}\n")
    
    # 1. MEC GEN
    print("⚙️  Ejecutando [ MEC GEN ] - Extracción de párrafos...")
    parrafos = mec_gen_extraer_parrafos(noticia_texto)
    print(f"👉 Se extrajeron {len(parrafos)} párrafos/oraciones para análisis individual:\n")
    for idx, p in enumerate(parrafos, 1):
        print(f"   ▫️ Párrafo {idx} (P{idx}): \"{p}\"")
    print("\n" + "-" * 80)
    
    # 2. MODELO CLASIFICADOR PRINCIPAL (SaBERT)
    print("🤖 [ MODELO CLASIFICADOR PRINCIPAL - SaBERT ] (Detección de Falsedad)")
    print("-" * 80)
    etiqueta_gen, prob_fake_gen, prob_true_gen = evaluar_sabert_falsedad(noticia_texto)
    print(f"📊 [ ANÁLISIS GENERAL ]")
    print(f"   - Noticia en General: [{etiqueta_gen}] (Prob. Falso: {prob_fake_gen:.4f} | Prob. Verdadero: {prob_true_gen:.4f})")
    
    print(f"\n📋 [ ANÁLISIS POR PÁRRAFO ]")
    evaluaciones_parrafos = []
    for idx, p in enumerate(parrafos, 1):
        lbl_p, p_fake, p_true = evaluar_sabert_falsedad(p)
        evaluaciones_parrafos.append({
            'id': idx,
            'nombre': f'P{idx}',
            'texto': p,
            'etiqueta': lbl_p,
            'score_fake': p_fake,
            'score_true': p_true
        })
        print(f"   - P{idx}: {lbl_p} (Score Falsedad: {p_fake:.4f})")
    print("\n" + "-" * 80)
    
    # 3. FILTRO DE PÁRRAFOS SCORE MÁS ALTO
    print("🎯 (Filtro: Pasan los párrafos con el SCORE MÁS ALTO de falsedad)")
    parrafos_ordenados = sorted(evaluaciones_parrafos, key=lambda x: x['score_fake'], reverse=True)
    parrafos_filtrados = [p for p in parrafos_ordenados if p['score_fake'] >= umbral_score_falsedad][:top_k_filtro]
    
    if not parrafos_filtrados and parrafos_ordenados:
        parrafos_filtrados = parrafos_ordenados[:top_k_filtro]
        
    parrafos_filtrados = sorted(parrafos_filtrados, key=lambda x: x['id'])
    
    print(f"👉 Párrafos Seleccionados ({len(parrafos_filtrados)}):")
    for pf in parrafos_filtrados:
        print(f"   ✅ {pf['nombre']} (Score Falsedad: {pf['score_fake']:.4f}): \"{pf['texto']}\"")
    print("\n" + "=" * 80)
    
    # 4. MODELOS SENSACIONALISMO Y REDUNDANCIA
    print(" [ PÁRRAFOS FILTRADOS + NOTICIA COMPLETA ENTRAN A MODELOS SECUNDARIOS ]")
    print("=" * 80 + "\n")
    
    print("=============================")
    print("[   MODELO CLASIFICADOR     ]")
    print("[     SENSACIONALISMO       ]")
    print("=============================")
    lbl_sens_gen, score_sens_gen = evaluar_sensacionalismo(noticia_texto)
    print(f"+-> Texto General: [{lbl_sens_gen}] (Score: {score_sens_gen:.4f})")
    for pf in parrafos_filtrados:
        lbl_sens_p, score_sens_p = evaluar_sensacionalismo(pf['texto'])
        print(f"+-> {pf['nombre']} ({pf['nombre']}): [{lbl_sens_p}] (Score: {score_sens_p:.4f})")
        
    print("\n" + "-" * 60 + "\n")
    
    print("=============================")
    print("[   MODELO CLASIFICADOR     ]")
    print("[       REDUNDANCIA         ]")
    print("=============================")
    res_redundancia = evaluar_redundancia(noticia_texto, parrafos)
    print(f"+-> Texto General: [{res_redundancia['global_label']}] (Score Similitud: {res_redundancia['global_score']:.4f})")
    for pf in parrafos_filtrados:
        idx_p = pf['id'] - 1
        score_red_p, lbl_red_p = res_redundancia['parrafos_scores'].get(idx_p, (0.0, "NO Redundante"))
        print(f"+-> {pf['nombre']} ({pf['nombre']}): [{lbl_red_p}] (Score Redundancia: {score_red_p:.4f})")
        
    print("\n" + "=" * 80)
    print("✨ PIPELINE FINALIZADO EXITOSAMENTE ✨")
    print("=" * 80)


# 🧪 6. Ejecución del Pipeline con Noticia de Ejemplo


In [ ]:
noticia_ejemplo = """
¡Escándalo total en la economía! Un informe confidencial filtrado por hackers revela que el Banco Central oculta la verdad y provocará la quiebra nacional mañana mismo.
Los analistas de la institución financiera señalan que las medidas tomadas buscan estabilizar los niveles de inflación gradualmente durante el próximo trimestre.
Expertos advierten que el colapso financiero es inminente y que el apocalipsis económico destruirá los ahorros de toda la población en cuestión de horas.
"""

ejecutar_pipeline_tesis(noticia_ejemplo)
